In [ ]:
import pandas as pd

output_csv  = r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\final_df_summaries.csv"
final_df    = pd.read_csv(output_csv)

# select rows where the summary is present
has_summary = final_df[ final_df["llama_generated_summary_from_truncated_text"].notna() ]

has_summary


In [ ]:
has_summary.columns
==
Index(['file_path', 'file_name', 'file_id', 'extension', 'original_text',
       'llama_generated_summary', 'box_file_path', 'size_bytes', 'created_at',
       'modified_at', 'is_empty', 'token_count_og_text', 'char_count_og_text',
       'pre_summary', 'truncated_original_text', 't_token_count_og_text',
       't_char_count_og_text', 'llama_generated_summary_from_truncated_text',
       'time_processed'],
      dtype='object')

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

# ─── 1) Load & filter your data ─────────────────────────────
output_csv = r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\final_df_summaries.csv"
df = pd.read_csv(output_csv)

# Keep only the rows with a summary
has_summary = df.dropna(subset=['llama_generated_summary_from_truncated_text']).reset_index(drop=True)
summaries   = has_summary['llama_generated_summary_from_truncated_text'].tolist()

# ─── 2) Embed with a SentenceTransformer ────────────────────
model = SentenceTransformer('all-MiniLM-L6-v2')
# This will give you a (N × d) NumPy array of embeddings
embeddings = model.encode(summaries, convert_to_numpy=True, show_progress_bar=True)

# ─── 3) Normalize for cosine similarity ────────────────────
faiss.normalize_L2(embeddings)

# ─── 4) Build the FAISS index ───────────────────────────────
d = embeddings.shape[1]                   # dimension of embeddings
index = faiss.IndexFlatIP(d)              # flat index, inner-product = cosine when normalized
index.add(embeddings)                     # add all summary vectors

# # ─── 5) Define a simple search function ────────────────────
# def search(query: str, k: int = 5):
#     # 1) embed and normalize the query
#     q_emb, = model.encode([query], convert_to_numpy=True)
#     faiss.normalize_L2(q_emb.reshape(1, -1))
#     # 2) retrieve top-k
#     D, I = index.search(q_emb.reshape(1, -1), k)
#     # 3) map back to text
#     results = [
#         {'score': float(score), 'summary': summaries[idx]}
#         for score, idx in zip(D[0], I[0])
#     ]
#     return results

# # ─── 6) Example ─────────────────────────────────────────────
# if __name__ == "__main__":
#     query = "Image of a protest?"
#     top_hits = search(query, k=5)
#     for hit in top_hits:
#         print(f"{hit['score']:.3f} ─ {hit['summary'][:100]}…")


0.697 ─ The text describes a photograph of a group of people protesting in a park, each holding signs with d…
0.696 ─ A protest image shows three individuals standing together, each holding a flag or banner expressing …
0.676 ─ The image shows a peaceful protest scene on a city sidewalk, where a group of people are holding sig…
0.656 ─ The image shows a group of people participating in a peaceful protest or rally in a park. The protes…
0.648 ─ A peaceful protest is taking place in an urban park, featuring two individuals holding contrasting s…


In [ ]:
# def search(query: str, k: int = 5):
#     # 1) embed & normalize query
#     q_emb = model.encode([query], convert_to_numpy=True)
#     faiss.normalize_L2(q_emb)
#     # 2) search
#     D, I = index.search(q_emb, k)
#     # 3) collect results
#     results = []
#     for score, idx in zip(D[0], I[0]):
#         results.append({
#             'score': float(score),
#             'box_file_path': has_summary.loc[idx, 'box_file_path'],
#             'summary': summaries[idx]
#         })
#     return results


In [ ]:
# hits = search("Image of a protest", k=10)
# for hit in hits:
#     print(hit['score'], hit['box_file_path'])
#     print(" →", hit['summary'], "\n")


In [ ]:
# import os

# def search(query: str, k: int = 5, threshold: float | None = None):
#     """
#     query      : your search string
#     k          : how many top hits to return (ignored if threshold is set)
#     threshold  : minimum similarity score to return (if set, returns *all* hits >= threshold)
#     """
#     # 1) embed & normalize the query
#     q_emb, = model.encode([query], convert_to_numpy=True)
#     faiss.normalize_L2(q_emb.reshape(1, -1))

#     # 2) retrieve top-(k or all) vectors
#     #    If threshold is set, pull *all* vectors and filter by score.
#     #    Otherwise just pull top-k.
#     if threshold is not None:
#         # get *all* distances by asking for index.ntotal neighbors
#         ntotal = index.ntotal
#         D, I = index.search(q_emb.reshape(1, -1), ntotal)
#     else:
#         D, I = index.search(q_emb.reshape(1, -1), k)

#     # 3) build full result list
#     results = []
#     for score, idx in zip(D[0], I[0]):
#         file_path = has_summary.loc[idx, 'box_file_path']
#         base      = os.path.basename(file_path)
#         name, ext = os.path.splitext(base)

#         results.append({
#             'score'       : float(score),
#             'box_file_path': file_path,
#             'file_name'   : name,
#             'extension'   : ext,
#             'summary'     : summaries[idx]
#         })

#     # 4) apply threshold filter if requested
#     if threshold is not None:
#         filtered = [r for r in results if r['score'] >= threshold]
#         # keep them sorted descending by score
#         return sorted(filtered, key=lambda r: r['score'], reverse=True)

#     # 5) otherwise just return the top-k
#     return results

# # ─── Example usage ───────────────────────────────────────────
# # top 5 hits:
# hits = search("Image of a protest", k=5)
# for hit in hits:
#     print(f"{hit['score']:.3f} — {hit['file_name']}{hit['extension']}")
#     print("  →", hit['summary'], "\n")

# # all hits with at least 0.80 similarity:
# hits_thresh = search("Image of a protest", threshold=0.50)
# print(f"{len(hits_thresh)} hits ≥ 0.50")


In [ ]:
# if __name__ == "__main__":
#     print("Semantic Search CLI (type 'exit' to quit)\n")

#     while True:
#         query = input("Enter your search query: ").strip()
#         if query.lower() in ('', 'exit', 'quit'):
#             print("Goodbye!")
#             break

#         # Ask: top-k or threshold?
#         mode = ''
#         while mode not in ('k', 't'):
#             mode = input("Search by (k) top-k or (t) threshold? [k/t]: ").strip().lower()

#         if mode == 'k':
#             try:
#                 k = int(input("Enter number of results (k): ").strip())
#             except ValueError:
#                 print("Invalid number, defaulting to k=5")
#                 k = 5
#             hits = search(query, k=k, threshold=None)

#         else:  # mode == 't'
#             try:
#                 threshold = float(input("Enter similarity threshold (0.0–1.0): ").strip())
#             except ValueError:
#                 print("Invalid threshold, defaulting to 0.75")
#                 threshold = 0.75
#             hits = search(query, k=None, threshold=threshold)

#         if not hits:
#             print("No results found.\n")
#             continue

# #         print(f"\nTop {len(hits)} result(s):")
# #         for i, hit in enumerate(hits, start=1):
# #             print(f"{i}. {hit['score']:.3f} — {hit['file_name']}{hit['extension']}")
# #             # show first 150 chars of summary
# #             snippet = hit['summary'].replace('\n',' ')
# #             print(f"    → {snippet[:150]}{'…' if len(snippet)>150 else ''}\n")
            
#         print(f"\nTop {len(hits)} result(s):")
#         for i, hit in enumerate(hits, start=1):
#             print(f"{i}. {hit['score']:.3f} — {hit['file_name']}{hit['extension']}")
#             print(f"   Path: {hit['box_file_path']}")
#             print(f"   {hit['summary']}\n")


#         print("-" * 40)


In [ ]:
print("🔍 Starting Semantic Search CLI…")


🔍 Starting Semantic Search CLI…


In [ ]:
import os
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import time

# ─── Preload data & build index ───────────────────────────────
print("🔍 Loading data and building FAISS index…")

output_csv = r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\final_df_summaries.csv"
df = pd.read_csv(output_csv)
has_summary = df.dropna(subset=['llama_generated_summary_from_truncated_text']).reset_index(drop=True)
summaries   = has_summary['llama_generated_summary_from_truncated_text'].tolist()

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(summaries, convert_to_numpy=True, show_progress_bar=False)
faiss.normalize_L2(embeddings)
d     = embeddings.shape[1]
index = faiss.IndexFlatIP(d)
index.add(embeddings)



In [ ]:
from PIL import Image

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tiff'}

def search(query: str, k: int = 5, threshold: float | None = None):
    q_emb, = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb.reshape(1, -1))

    if threshold is not None:
        ntotal = index.ntotal
        D, I = index.search(q_emb.reshape(1, -1), ntotal)
    else:
        D, I = index.search(q_emb.reshape(1, -1), k)

    results = []
    for score, idx in zip(D[0], I[0]):
        file_path = has_summary.loc[idx, 'box_file_path']
        base      = os.path.basename(file_path)
        name, ext = os.path.splitext(base)
        results.append({
            'score': float(score),
            'df_index': idx,
            'box_file_path': file_path,
            'file_name': name,
            'extension': ext,
            'summary': summaries[idx]
        })
    if threshold is not None:
        results = [r for r in results if r['score'] >= threshold]
        results.sort(key=lambda r: r['score'], reverse=True)
    return results

# ─── Interactive loop ────────────────────────────────────────
print("🔍 Starting Semantic Search CLI (type 'exit' to quit)\n")

while True:
    query = input("Search query: ").strip()
    if not query or query.lower() in ('exit','quit'):
        print("👋 Goodbye!")
        break

    mode = ''
    while mode not in ('k','t'):
        mode = input("Search by (k) top-k or (t) threshold? [k/t]: ").strip().lower()

    if mode == 'k':
        try:
            k = int(input("Number of results (k): ").strip())
        except:
            print("Invalid, defaulting to k=5")
            k = 5
        hits = search(query, k=k)
    else:
        try:
            threshold = float(input("Similarity threshold (0.0–1.0): ").strip())
        except:
            print("Invalid, defaulting to 0.75")
            threshold = 0.75
        hits = search(query, threshold=threshold)

    if not hits:
        print("No results found.\n")
        continue

    print(f"\nTop {len(hits)} result(s):")
    for i, hit in enumerate(hits, 1):
        print(f"{i}. {hit['score']:.3f} — {hit['file_name']}{hit['extension']}")
        print(f"   Path: {hit['box_file_path']}")
        print(f"   {hit['summary'][:500]}…\n")

#     drill = input("Enter result number to see full text (or Enter to skip): ").strip()
#     if drill.isdigit():
#         sel = int(drill)
#         if 1 <= sel <= len(hits):
#             row_idx = hits[sel-1]['df_index']
#             full_text = has_summary.at[row_idx, 'truncated_original_text']
#             print("\n── Full original text ──────────────────")
#             print(full_text)
#             print("──────────────────────────────────────────\n")
#             input("Press Enter to continue…")
#     print()

    drill = input("Enter result number to see full text/image (or Enter to skip): ").strip()
    if drill.isdigit():
        sel = int(drill)
        if 1 <= sel <= len(hits):
            hit = hits[sel-1]
            row_idx = hit['df_index']

            # local file system path
            local_path = has_summary.at[row_idx, 'file_path']
            ext        = has_summary.at[row_idx, 'extension'].lower()

            if ext in IMAGE_EXTS:
                print(f"\nOpening image: {local_path}\n")
                try:
                    img = Image.open(local_path)
                    img.show()    # will launch your OS default image viewer
                except Exception as e:
                    print(f"❌ Could not open image: {e}")

            else:
                # print the full original text
                full_text = has_summary.at[row_idx, 'truncated_original_text']
                print("\n── Full original text ──────────────────")
                print(full_text)
                print("──────────────────────────────────────────\n")

            input("Press Enter to continue…")
        else:
            print("⚠️  Invalid selection.\n")


In [ ]:
import os
from PIL import Image
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
import time

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tiff'}

# ─── Preload data & build index ───────────────────────────────
output_csv = r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\final_df_summaries.csv"
df         = pd.read_csv(output_csv)
has_summary = df.dropna(subset=['llama_generated_summary_from_truncated_text']).reset_index(drop=True)
summaries   = has_summary['llama_generated_summary_from_truncated_text'].tolist()

model      = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(summaries, convert_to_numpy=True, show_progress_bar=False)
faiss.normalize_L2(embeddings)
d     = embeddings.shape[1]
index = faiss.IndexFlatIP(d)
index.add(embeddings)

def search(query: str, k: int = 5, threshold: float | None = None):
    q_emb, = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb.reshape(1, -1))

    if threshold is not None:
        D, I = index.search(q_emb.reshape(1, -1), index.ntotal)
    else:
        D, I = index.search(q_emb.reshape(1, -1), k)

    results = []
    for score, idx in zip(D[0], I[0]):
        box_path = has_summary.at[idx, 'box_file_path']
        base     = os.path.basename(box_path)
        name, ext = os.path.splitext(base)
        results.append({
            'score'        : float(score),
            'df_index'     : idx,
            'box_file_path': box_path,
            'file_name'    : name,
            'extension'    : ext.lower(),
            'summary'      : summaries[idx]
        })
    if threshold is not None:
        results = [r for r in results if r['score'] >= threshold]
        results.sort(key=lambda r: r['score'], reverse=True)
    return results

if __name__ == "__main__":
    print("🔍 Starting Semantic Search CLI (type 'exit' to quit)\n")

    while True:
        query = input("Search query: ").strip()
        if not query or query.lower() in ('exit','quit'):
            print("👋 Goodbye!")
            break

        # choose top-k vs threshold
        mode = ''
        while mode not in ('k','t'):
            mode = input("Search by (k) top-k or (t) threshold? [k/t]: ").strip().lower()

        if mode == 'k':
            try:
                k = int(input("Number of results (k): ").strip())
            except:
                print("Invalid, defaulting to k=5")
                k = 5
            hits = search(query, k=k)
        else:
            try:
                threshold = float(input("Similarity threshold (0.0–1.0): ").strip())
            except:
                print("Invalid, defaulting to 0.75")
                threshold = 0.75
            hits = search(query, threshold=threshold)

        if not hits:
            print("No results found.\n")
            continue

        print(f"\nTop {len(hits)} result(s):")
        for i, hit in enumerate(hits, 1):
            print(f"{i}. {hit['score']:.3f} — {hit['file_name']}{hit['extension']}")
            print(f"    Path: {hit['box_file_path']}")
            print(f"    {hit['summary'][:500]}…\n")

        choice = input("Enter result number to see full text/image (or Enter to skip): ").strip()
        if choice.isdigit():
            sel = int(choice)
            if 1 <= sel <= len(hits):
                hit      = hits[sel-1]
                row_idx  = hit['df_index']
                local_fp = has_summary.at[row_idx, 'file_path']
                ext      = hit['extension']

                if ext in IMAGE_EXTS:
                    print(f"\nOpening image: {local_fp}\n")
                    try:
                        Image.open(local_fp).show()
                    except Exception as e:
                        print(f"❌ Could not open image: {e}")
                else:
                    full_text = has_summary.at[row_idx, 'truncated_original_text']
                    print("\n── Full original text ──────────────────")
                    print(full_text)
                    print("──────────────────────────────────────────\n")

                input("Press Enter to continue…")
            else:
                print("⚠️ Invalid selection.\n")

        print()


🔍 Starting Semantic Search CLI (type 'exit' to quit)



In [ ]:
import os
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

# ─── Config ─────────────────────────────────────────────────────────
input_csv  = r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\final_df_summaries.csv"
output_csv = input_csv.replace(".csv", "_with_topics.csv")

ollama_path = r"C:\Users\terbe\AppData\Local\Programs\Ollama\ollama.exe"
model_name  = "llama3.2"

# ─── 0) Load your summaries ────────────────────────────────────────
df = pd.read_csv(input_csv)
texts = df['llama_generated_summary_from_truncated_text'].fillna("").tolist()

# ─── 1) Vectorize with TF-IDF ──────────────────────────────────────
vectorizer = TfidfVectorizer(
    max_df=0.9,      # drop super‐common words
    min_df=5,        # drop very rare words
    stop_words='english',
    ngram_range=(1,2)
)
X = vectorizer.fit_transform(texts)
terms = np.array(vectorizer.get_feature_names_out())

# ─── 2) Elbow method to pick K ────────────────────────────────────
inertias = []
K_range = range(2, 16)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(X)
    inertias.append(km.inertia_)

plt.plot(K_range, inertias, 'o-')
plt.xlabel("Number of topics (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for Optimal K")
plt.show()



In [ ]:
# → Inspect the plot, pick your n_topics (e.g. elbow at 6)
n_topics = 6

# ─── 3) Fit final KMeans ──────────────────────────────────────────
km = KMeans(n_clusters=n_topics, random_state=42)
clusters = km.fit_predict(X)
df['topic_id'] = clusters

# ─── 4) Extract top keywords per topic ───────────────────────────
# For each cluster center, take the top N terms by weight
N_key = 20
topic_keywords = {}
for tid, center in enumerate(km.cluster_centers_):
    top_idx = center.argsort()[::-1][:N_key]
    topic_keywords[tid] = terms[top_idx].tolist()

# ─── 5) Ask Llama for a title per topic ───────────────────────────
def llama_title_for_keywords(keywords: list[int], topic_id: int) -> str:
    prompt = (
        "Given these topic keywords:\n\n"
        f"{', '.join(keywords)}\n\n"
        "Write a concise, <1-sentence title summarizing this topic."
    )
    proc = subprocess.run(
        [ollama_path, "run", model_name, "--", prompt],
        capture_output=True, text=False
    )
    out = proc.stdout.decode('utf-8', errors='replace').strip()
    if proc.returncode != 0:
        err = proc.stderr.decode('utf-8', errors='replace')
        print(f"❌ Llama error for topic {topic_id}: {err}")
        return ""
    return out

# Build a DataFrame of topic metadata
topics = []
for tid, keys in topic_keywords.items():
    title = llama_title_for_keywords(keys, tid)
    topics.append({
        'topic_id': tid,
        'keywords': keys,
        'topic_title': title
    })
topics_df = pd.DataFrame(topics)



In [ ]:
topics_df

In [ ]:
# ─── 6) Merge topic info back onto your summaries ────────────────
result = df.merge(topics_df, on='topic_id', how='left')
result



In [ ]:
result["topic_title"].value_counts()

topic_title
Visualizing Data: A Guide to Creating Effective Graphs and Diagrams.                                             150
"Unraveling the Dynamics of Gas-Liquid Flows and Turbulence: A Study on Fluid Behavior in Slug-Flow Regimes."    124
"Crafting a Striking Image: Mastering the Art of Composition and Color on a Gray Background."                    101
"The Capturing of a Scene: A Photograph's Depiction of People in a Group Setting."                                79
"University of Illinois Intellectual Property and Technology Transfer Committee Report Summary".                  52
"Assisting with Text Summarization: Providing Accurate and Helpful Output."                                       48
Name: count, dtype: int64
        
viz_data = result[result["topic_title"] == "Visualizing Data: A Guide to Creating Effective Graphs and Diagrams."]
viz_data        

In [ ]:
viz_data = result[result["topic_title"] == "Visualizing Data: A Guide to Creating Effective Graphs and Diagrams."]
viz_data

In [ ]:
# ─── 7) Save out ─────────────────────────────────────────────────
result.to_csv(output_csv, index=False)
print("Saved with topics to", output_csv)

Saved with topics to C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\final_df_summaries_with_topics.csv


In [ ]:
import subprocess
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

# ─── Config ─────────────────────────────────────────────────────────
input_csv     = r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\final_df_summaries_with_topics.csv"
output_nested = input_csv.replace(".csv", "_nested.csv")

ollama_path = r"C:\Users\terbe\AppData\Local\Programs\Ollama\ollama.exe"
model_name  = "llama3.2"

# How many subtopics per parent-topic?
n_subtopics = 3
# How many keywords to extract per subtopic
N_key       = 10

# ─── Helpers ────────────────────────────────────────────────────────
def llama_title_for_keywords(keywords: list[str], parent: int, child: int) -> str:
    prompt = (
        f"These are keywords from subtopic {child} of main topic {parent}:\n\n"
        + ", ".join(keywords)
        + "\n\n Give only a concise one-sentence title with no other explanation."
    )
    proc = subprocess.run(
        [ollama_path, "run", model_name, "--", prompt],
        capture_output=True, text=False
    )
    if proc.returncode != 0:
        err = proc.stderr.decode('utf-8', errors='replace')
        print(f"❌ Llama error for subtopic {parent}.{child}: {err}")
        return ""
    return proc.stdout.decode('utf-8', errors='replace').strip()

# ─── 0) Load your data with top‐level topics ───────────────────────
result = pd.read_csv(input_csv)

nested_frames = []

# ─── 1) Loop over each parent topic ──────────────────────────────
for parent_id, df_parent in result.groupby('topic_id', sort=True):
    texts = df_parent['llama_generated_summary_from_truncated_text']\
                .fillna("").tolist()

    # 1a) Vectorize
    vect = TfidfVectorizer(
        max_df=0.9, min_df=2,
        stop_words='english',
        ngram_range=(1,2)
    )
    Xt = vect.fit_transform(texts)
    terms = np.array(vect.get_feature_names_out())

    # 1b) KMeans for subtopics
    km = KMeans(n_clusters=n_subtopics, random_state=42)
    sub_labels = km.fit_predict(Xt)
    df_parent = df_parent.copy()
    df_parent['subtopic_id'] = sub_labels

    # 1c) Extract keywords per subtopic center
    sub_keywords = {}
    for sid, center in enumerate(km.cluster_centers_):
        top_idx = center.argsort()[::-1][:N_key]
        sub_keywords[sid] = terms[top_idx].tolist()

    # 1d) Generate titles from Llama
    subtopic_meta = []
    for sid, kws in sub_keywords.items():
        title = llama_title_for_keywords(kws, parent_id, sid)
        subtopic_meta.append({
            'topic_id'        : parent_id,
            'subtopic_id'     : sid,
            'subtopic_keywords': kws,
            'subtopic_title'  : title
        })
    subtopics_df = pd.DataFrame(subtopic_meta)

    # 1e) Merge titles back onto this slice
    df_parent = df_parent.merge(
        subtopics_df,
        on=['topic_id','subtopic_id'],
        how='left'
    )

    nested_frames.append(df_parent)

# ─── 2) Combine all slices & save ─────────────────────────────────
nested_result = pd.concat(nested_frames, ignore_index=True)
nested_result.to_csv(output_nested, index=False)

print(f"Nested topics saved to:\n  {output_nested}")


Nested topics saved to:
  C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\final_df_summaries_with_topics_nested.csv


In [ ]:
nested_result['subtopic_title'].value_counts()

In [ ]:
import math
import subprocess
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

ollama_path = r"C:\Users\terbe\AppData\Local\Programs\Ollama\ollama.exe"
model_name  = "llama3.2"

max_size = 5
N_key    = 10

def llama_title_for_keywords(keywords: list[str], path: str) -> str:
    prompt = (
        f"These are keywords for topic {path}:\n\n"
        + ", ".join(keywords)
        + "\n\n Provide only a concise one-sentence title with no other explanation."
    )
    proc = subprocess.run(
        [ollama_path, "run", model_name, "--", prompt],
        capture_output=True, text=False
    )
    if proc.returncode != 0:
        err = proc.stderr.decode('utf-8', errors='replace')
        print(f"❌ Llama error at {path}: {err}")
        return ""
    return proc.stdout.decode('utf-8', errors='replace').strip()

def recursive_cluster(df: pd.DataFrame,
                      text_col: str,
                      path: str = "",
                      results: list[dict] = None) -> pd.DataFrame:
    if results is None:
        results = []

    size = len(df)
    texts = df[text_col].fillna("").tolist()

    # --- terminal cluster if small enough
    if size <= max_size:
        # if there's only one doc, just pick its top tokens by frequency
        if size == 1:
            words = texts[0].split()
            freq  = pd.Series(words).value_counts()
            keywords = freq.index.tolist()[:N_key]
        else:
            # dynamic TF-IDF settings
            min_df = 1
            max_df = 0.9 if size > 10 else 1.0

            vect = TfidfVectorizer(
                min_df = min_df,
                max_df = max_df,
                stop_words = 'english',
                ngram_range = (1,2)
            )
            X     = vect.fit_transform(texts)
            terms = np.array(vect.get_feature_names_out())

            # average to get a “centroid”
            center  = X.mean(axis=0).A1
            top_idx = center.argsort()[::-1][:N_key]
            keywords = terms[top_idx].tolist()

        title = llama_title_for_keywords(keywords, path or "root")
        for idx in df.index:
            results.append({
                'index'       : idx,
                'topic_path'  : path or "root",
                'keywords'    : keywords,
                'topic_title' : title
            })

    # --- split further otherwise
    else:
        n_clusters = math.ceil(size / max_size)

        # vectorize with the same dynamic params
        min_df = 1
        max_df = 0.9 if size > 10 else 1.0
        vect = TfidfVectorizer(
            min_df = min_df,
            max_df = max_df,
            stop_words = 'english',
            ngram_range = (1,2)
        )
        X = vect.fit_transform(texts)

        km     = KMeans(n_clusters=n_clusters, random_state=42)
        labels = km.fit_predict(X)

        for sub_id in range(n_clusters):
            sub_df  = df[labels == sub_id]
            sub_path = f"{path}.{sub_id}" if path else str(sub_id)
            recursive_cluster(sub_df, text_col, sub_path, results)

    return pd.DataFrame(results)


In [ ]:
# if __name__ == "__main__":
#     input_csv = r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\final_df_summaries_with_topics.csv"
#     df        = pd.read_csv(input_csv)

#     meta_df   = recursive_cluster(df, text_col='llama_generated_summary_from_truncated_text')
#     final     = df.merge(meta_df, left_index=True, right_on='index', how='left')

#     out_csv   = input_csv.replace(".csv", "_hierarchical.csv")
#     final.to_csv(out_csv, index=False)
#     print("Saved to", out_csv)


KeyboardInterrupt: 

In [ ]:
final